### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection.

In [ ]:
### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

* Long-running conversations that exceed context windows.
* Multi-turn dialogues with extensive history.
* Applications where preserving full conversation context matters.

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

agent = create_agent(
    model="ollama:granite4.1:3b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="ollama:tinyllama:1.1b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ],
)

In [13]:
### Run with a thread id
config={"configurable": {"thread_id": "test-1"}}

In [14]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='1b344af2-2af1-4421-ab36-550d4d27e27b'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'model': 'granite4.1:3b', 'created_at': '2026-05-25T14:04:07.1592516Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4517326900, 'load_duration': 3433441400, 'prompt_eval_count': 15, 'prompt_eval_duration': 358039600, 'eval_count': 9, 'eval_duration': 681282000, 'logprobs': None, 'model_name': 'granite4.1:3b', 'model_provider': 'ollama'}, id='lc_run--019e5f73-26ce-7470-81f2-976ee5e74ab1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 9, 'total_tokens': 24})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='1b344af2-2af1-4421-ab36-550d4d27e27b'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'model': 'granite4.1:3b

Summarization based on token size.

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="ollama:granite4.1:3b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="ollama:tinyllama:1.1b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [16]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~152 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='2030f6a1-a21c-499c-af35-11aa59483db1'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'granite4.1:3b', 'created_at': '2026-05-25T14:38:33.4954761Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7535628300, 'load_duration': 1413459100, 'prompt_eval_count': 174, 'prompt_eval_duration': 4329839700, 'eval_count': 21, 'eval_duration': 1655170200, 'logprobs': None, 'model_name': 'granite4.1:3b', 'model_provider': 'ollama'}, id='lc_run--019e5f92-a2a6-7e53-aa6e-1e567aa01dd8-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '1cbe328b-d9bd-4967-8bde-c150be9c2fba', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 174, 'output_tokens': 21, 'total_tokens': 195}), ToolMessage(content='Hotels in Paris:\n    1. Grand Hotel - 5 star, $350/night, spa, pool, gym\n    2. City Inn - 4 star, $180